# Distribution OPF (ADMM) — Analysis

This notebook post-processes and analyzes simulation results from the PNNL ADMM DOPF federate.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from oedisi.types.data_types import Topology
from admm_federate.adapter import area_disconnects, disconnect_areas, generate_graph
from admm_federate.plotting import (
    load_scenario_parameters,
    get_der_mapping,
    load_recorder_data,
    process_voltages,
    process_power_flows,
    process_generation_adequacy,
    process_convergence,
    plot_network_partition,
    plot_voltage_comparison,
    plot_power_flow_comparison,
    plot_generation_adequacy,
    plot_algorithmic_convergence,
)

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

In [ ]:
build_dir = Path(DATA_DIR) if DATA_DIR else Path(".").resolve()
run_dir = build_dir.parent if build_dir.name == "build" else build_dir
wiring_path = run_dir / "wiring.json"
outputs_dir = run_dir / "outputs"
if not outputs_dir.exists():
    outputs_dir = build_dir

if wiring_path.exists():
    with open(wiring_path) as f:
        wiring = json.load(f)
    area_ids, area_params = load_scenario_parameters(wiring)
    print(f"Loaded {len(area_ids)} control areas: {area_ids}")
else:
    print("Wiring diagram not found, using default empty config")
    wiring = {}
    area_ids, area_params = [], []

In [ ]:
topology_path = outputs_dir / "topology.json"
if not topology_path.exists():
    for p in run_dir.rglob("topology.json"):
        topology_path = p
        break

if topology_path.exists():
    topology = Topology.model_validate_json(topology_path.read_text(encoding="utf-8"))
    slack_bus = topology.slack_bus[0].split(".", 1)[0]
    G = generate_graph(topology.incidences, slack_bus)

    graph_for_partition = G.copy()
    graph_for_split = G.copy()
    boundaries = area_disconnects(graph_for_partition, n_max=len(area_ids))
    areas_clean = disconnect_areas(graph_for_split, boundaries)

    area_buses = [list(area.nodes()) for area in areas_clean]
    der_map = get_der_mapping(topology_path)

    coords_dir = build_dir
    for p in run_dir.rglob("Buscoords.*"): 
        coords_dir = p.parent
        break

    fig_partition = plot_network_partition(G, boundaries, areas_clean, slack_bus, coords_dir, target="notebook")
    if fig_partition:
        plt.show()
else:
    print("topology.json not found in simulation outputs")

In [ ]:
if area_ids and outputs_dir.exists():
    data = load_recorder_data(outputs_dir, wiring)
    voltage_data = process_voltages(data, area_ids, area_buses, topology)
    fig_volt = plot_voltage_comparison(voltage_data, target="notebook")
    if fig_volt:
        plt.show()

    flow_data = process_power_flows(data, area_ids, area_params, G, area_buses, der_map, slack_bus)
    fig_flow = plot_power_flow_comparison(flow_data, target="notebook")
    if fig_flow:
        plt.show()

    adequacy_df = process_generation_adequacy(topology, area_ids, area_buses)
    fig_adeq = plot_generation_adequacy(adequacy_df, target="notebook")
    if fig_adeq:
        plt.show()

    convergence_data = process_convergence(data, area_ids)
    fig_conv = plot_algorithmic_convergence(convergence_data, target="notebook")
    if fig_conv:
        plt.show()